# Explore all PBMC3k files

This notebook gives complete, usable access to the PBMC3k count matrix, BAM reads, BAM index, HDF5 molecule data, and Cell Ranger analysis tables.

**Important:** BAM, BAI, and H5 files are binary data formats, not archives. They should be opened with the appropriate library rather than extracted. The BAM alone is about 16 GB, so rendering every record in one browser page would freeze or crash Jupyter. The functions below provide pagination and region queries over the complete files—change the start, stop, and limit values to inspect any part.

## What each file is for

| Data | Main use |
|---|---|
| `filtered_gene_bc_matrices/hg19` | Main gene-by-cell count matrix for most single-cell analysis |
| `raw_gene_bc_matrices/hg19` | Counts for every detected barcode, including empty droplets |
| `.bam` | Individual aligned sequencing reads; useful for read-level or genome-browser work |
| `.bai` | Index used automatically to jump to regions of the BAM; it is not analyzed by itself |
| `molecule_info.h5` | Barcode, UMI, gene, and read-count data for individual molecules |
| `analysis/` CSV files | Cell Ranger PCA, t-SNE, clusters, and differential-expression results |
| `metrics_summary.csv` | Run-level quality-control summary |
| `web_summary.html` | Interactive Cell Ranger report; open directly in a web browser |

In [1]:
from pathlib import Path
from itertools import islice

import h5py
import numpy as np
import pandas as pd
import pysam
from scipy.io import mmread
from IPython.display import display

# Works whether Jupyter starts in PBMC3k/ or PBMC3k/notebooks/.
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if not (PROJECT / "data" / "raw").exists():
    raise FileNotFoundError("Start Jupyter from the PBMC3k directory or its notebooks directory.")

RAW = PROJECT / "data" / "raw"
INTERIM = PROJECT / "data" / "interim"
FILTERED = RAW / "filtered_gene_bc_matrices" / "hg19"

print("Project:", PROJECT)
print("Raw data:", RAW)

Project: /Users/Ema/Desktop/cosmos/26-the-backpropagators-analysis/PBMC3k
Raw data: /Users/Ema/Desktop/cosmos/26-the-backpropagators-analysis/PBMC3k/data/raw


## File inventory and validation

This lists every raw-data file and its size. A size is shown for every file; directory sizes are intentionally omitted.

In [2]:
def human_size(size):
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(size)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:,.1f} {unit}"
        value /= 1024

inventory = pd.DataFrame(
    [(str(p.relative_to(RAW)), human_size(p.stat().st_size)) for p in sorted(RAW.rglob("*")) if p.is_file()],
    columns=["file", "size"],
)
pd.set_option("display.max_rows", None)
display(inventory)
pd.reset_option("display.max_rows")

,file,size
0,.DS_Store,6.0 KB
1,.gitkeep,0.0 B
2,analysis/kmeans/10_clusters/clusters.csv,53.0 KB
3,analysis/kmeans/10_clusters/differential_expre...,5.4 MB
4,analysis/kmeans/2_clusters/clusters.csv,52.8 KB
5,analysis/kmeans/2_clusters/differential_expres...,1.6 MB
6,analysis/kmeans/3_clusters/clusters.csv,52.8 KB
7,analysis/kmeans/3_clusters/differential_expres...,2.2 MB
8,analysis/kmeans/4_clusters/clusters.csv,52.8 KB
9,analysis/kmeans/4_clusters/differential_expres...,2.6 MB


## Complete filtered gene-expression matrix

The complete sparse matrix is loaded into memory without expanding its mostly-zero values. Rows in the source matrix are genes and columns are cells. `expression_page` displays any rectangular portion as cells × genes.

In [3]:
genes = pd.read_csv(FILTERED / "genes.tsv", sep="\t", header=None, names=["gene_id", "gene_name"])
barcodes = pd.read_csv(FILTERED / "barcodes.tsv", sep="\t", header=None, names=["barcode"])
counts = mmread(FILTERED / "matrix.mtx").tocsr()

assert counts.shape == (len(genes), len(barcodes))
print(f"Complete matrix: {counts.shape[0]:,} genes × {counts.shape[1]:,} cells")
print(f"Nonzero expression values: {counts.nnz:,}")

Complete matrix: 32,738 genes × 2,700 cells
Nonzero expression values: 2,286,884


In [4]:
def expression_page(cell_start=0, cell_count=25, gene_start=0, gene_count=30):
    """Display any page of the complete count matrix."""
    cell_stop = min(cell_start + cell_count, counts.shape[1])
    gene_stop = min(gene_start + gene_count, counts.shape[0])
    page = counts[gene_start:gene_stop, cell_start:cell_stop].T.toarray()
    return pd.DataFrame(
        page,
        index=barcodes.iloc[cell_start:cell_stop]["barcode"],
        columns=genes.iloc[gene_start:gene_stop]["gene_name"],
    )

def find_genes(text):
    """Find all genes whose ID or symbol contains text."""
    mask = genes.astype(str).apply(lambda col: col.str.contains(text, case=False, regex=False)).any(axis=1)
    return genes.loc[mask]

display(expression_page(cell_start=0, cell_count=25, gene_start=0, gene_count=30))
display(find_genes("CD3D"))

gene_name,MIR1302-10,FAM138A,OR4F5,RP11-34P13.7,RP11-34P13.8,AL627309.1,RP11-34P13.14,RP11-34P13.9,AP006222.2,RP4-669L17.10,...,RP11-206L10.5,RP11-206L10.4,RP11-206L10.2,RP11-206L10.9,AL669831.1,FAM87B,LINC00115,FAM41C,AL645608.2,RP11-54O7.16
barcode,,,,,,,,,,,,,,,,,,,,,
AAACATACAACCAC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACATTGAGCTAC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACATTGATCAGC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCGTGCTTCCG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACCGTGTATGCG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGCACTGGTAC-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGCTGACCAGT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGCTGGTTCTT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AAACGCTGTAGCCA-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


,gene_id,gene_name
19801,ENSG00000167286,CD3D


Change the four arguments above to view any cells or genes. For example, `expression_page(cell_start=2500, cell_count=100, gene_start=30000, gene_count=100)` shows a later page. Avoid requesting the entire dense matrix at once; keep `counts` sparse for analysis.

## BAM and BAI: complete aligned-read access

The links in `data/interim` give the BAM and its index the matching names expected by common software. The original raw files remain unchanged. Use `bam_page` to inspect reads from any chromosome or region. Coordinates are zero-based, half-open.

In [5]:
BAM_PATH = INTERIM / "pbmc3k.bam"
bam = pysam.AlignmentFile(BAM_PATH, "rb")

print("References:", bam.nreferences)
display(pd.DataFrame({"reference": bam.references, "length": bam.lengths}).head(20))
print("Header excerpt:")
print(str(bam.header).splitlines()[0:10])

References: 84


,reference,length
0,1,249250621
1,10,135534747
2,11,135006516
3,12,133851895
4,13,115169878
5,14,107349540
6,15,102531392
7,16,90354753
8,17,81195210
9,18,78077248


Header excerpt:
['@HD\tVN:1.4\tSO:coordinate', '@SQ\tSN:1\tLN:249250621', '@SQ\tSN:10\tLN:135534747', '@SQ\tSN:11\tLN:135006516', '@SQ\tSN:12\tLN:133851895', '@SQ\tSN:13\tLN:115169878', '@SQ\tSN:14\tLN:107349540', '@SQ\tSN:15\tLN:102531392', '@SQ\tSN:16\tLN:90354753', '@SQ\tSN:17\tLN:81195210']


In [6]:
def bam_page(reference=None, start=None, stop=None, offset=0, limit=100):
    """Return a page of BAM alignments, optionally restricted to a genomic region."""
    if reference is None:
        bam.reset()
        records = bam.fetch(until_eof=True)
    else:
        records = bam.fetch(reference, start, stop)
    selected = islice(records, offset, offset + limit)
    rows = []
    for read in selected:
        rows.append({
            "read_name": read.query_name,
            "reference": read.reference_name,
            "start": read.reference_start,
            "end": read.reference_end,
            "mapq": read.mapping_quality,
            "cigar": read.cigarstring,
            "cell_barcode": read.get_tag("CB") if read.has_tag("CB") else None,
            "umi": read.get_tag("UB") if read.has_tag("UB") else None,
            "gene": read.get_tag("GX") if read.has_tag("GX") else None,
        })
    return pd.DataFrame(rows)

# First 100 records. Increase offset/limit or specify a reference and coordinates.
display(bam_page(limit=100))
# Example regional query (uncomment and use a reference name shown above):
# display(bam_page(reference="1", start=1_000_000, stop=1_100_000, limit=200))

,read_name,reference,start,end,mapq,cigar,cell_barcode,umi,gene
0,D00547:633:HKMNVBCXX:2:1112:6541:82867,1,11853,11920,1,31S67M,TGATTAGATGACTG-1,CCCGGTATAG,None
1,D00547:633:HKMNVBCXX:2:1106:8453:55346,1,11853,11921,1,30S68M,TGCGAAACAGTCAC-1,AGCCAGTCGT,None
2,D00547:633:HKMNVBCXX:2:1115:15507:59792,1,12045,12143,0,98M,CCTCTACTCTTCGC-1,ATATTTACAG,None
3,D00547:633:HKMNVBCXX:2:1213:11344:50551,1,12149,12632,3,78M385N20M,None,CAGGATCACA,None
4,D00547:633:HKMNVBCXX:1:1108:9950:86956,1,13181,13279,1,98M,TTCTCAGATGGAGG-1,ACCTTCCCCC,None
...,...,...,...,...,...,...,...,...,...
95,D00547:633:HKMNVBCXX:2:1114:1212:72110,1,14100,14198,1,98M,ACCCAAGAATTCCT-1,TCGTATCTTC,None
96,D00547:633:HKMNVBCXX:1:2104:5093:23819,1,14100,14198,1,98M,ATCATCTGACACCA-1,TACAACCTTT,None
97,D00547:633:HKMNVBCXX:2:2116:17561:20437,1,14100,14198,1,98M,CATAAATGTCTGGA-1,TGCACTGCGT,None
98,D00547:633:HKMNVBCXX:1:2208:3555:23662,1,14101,14197,1,2S96M,AACTCGGATGCCTC-1,ACAAAAGATG,None


`bam.count(until_eof=True)` can count every alignment, but it must scan the full 16 GB file and may take a long time. Genome browsers such as IGV can also open `data/interim/pbmc3k.bam`; the `.bai` link will be detected automatically.

## HDF5 molecule information: every dataset and paginated values

`h5_catalog` lists every group and dataset in the file. `h5_page` reads any one-dimensional dataset by row range without loading the entire 594 MB file.

In [7]:
H5_PATH = RAW / "pbmc3k_molecule_info.h5"
h5 = h5py.File(H5_PATH, "r")

catalog_rows = []
def catalog_item(name, obj):
    catalog_rows.append({
        "name": name,
        "type": type(obj).__name__,
        "shape": getattr(obj, "shape", None),
        "dtype": str(getattr(obj, "dtype", "")),
    })

h5.visititems(catalog_item)
h5_catalog = pd.DataFrame(catalog_rows)
pd.set_option("display.max_rows", None)
display(h5_catalog)
pd.reset_option("display.max_rows")

,name,type,shape,dtype
0,barcode,Dataset,"(15185957,)",uint64
1,barcode_corrected_reads,Dataset,"(15185957,)",uint32
2,conf_mapped_uniq_read_pos,Dataset,"(15185957,)",uint32
3,gem_group,Dataset,"(15185957,)",uint8
4,gene,Dataset,"(15185957,)",uint32
5,nonconf_mapped_reads,Dataset,"(15185957,)",uint32
6,reads,Dataset,"(15185957,)",uint32
7,umi,Dataset,"(15185957,)",uint32
8,umi_corrected_reads,Dataset,"(15185957,)",uint32
9,unmapped_reads,Dataset,"(15185957,)",uint32


In [8]:
def h5_page(dataset_name, start=0, limit=100):
    """Read a page from any HDF5 dataset listed in h5_catalog."""
    dataset = h5[dataset_name]
    if not isinstance(dataset, h5py.Dataset):
        raise TypeError(f"{dataset_name!r} is a group, not a dataset")
    if dataset.ndim == 0:
        return dataset[()]
    values = dataset[start:min(start + limit, dataset.shape[0])]
    if values.ndim == 1:
        return pd.Series(values, name=dataset_name, index=range(start, start + len(values)))
    return values

# Pick any dataset name displayed in h5_catalog, then run:
# display(h5_page("reads", start=0, limit=100))

## Cell Ranger CSV results

List every CSV and load any complete table. These tables are much smaller than the BAM and can normally be displayed in full.

In [9]:
csv_files = sorted(RAW.rglob("*.csv"))
csv_catalog = pd.DataFrame({"number": range(len(csv_files)), "file": [str(p.relative_to(RAW)) for p in csv_files]})
pd.set_option("display.max_rows", None)
display(csv_catalog)
pd.reset_option("display.max_rows")

def open_csv(number):
    """Load a complete CSV selected by its catalog number."""
    return pd.read_csv(csv_files[number])

# Example: display every row of the first CSV.
table = open_csv(0)
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(table)

,number,file
0,0,analysis/kmeans/10_clusters/clusters.csv
1,1,analysis/kmeans/10_clusters/differential_expre...
2,2,analysis/kmeans/2_clusters/clusters.csv
3,3,analysis/kmeans/2_clusters/differential_expres...
4,4,analysis/kmeans/3_clusters/clusters.csv
5,5,analysis/kmeans/3_clusters/differential_expres...
6,6,analysis/kmeans/4_clusters/clusters.csv
7,7,analysis/kmeans/4_clusters/differential_expres...
8,8,analysis/kmeans/5_clusters/clusters.csv
9,9,analysis/kmeans/5_clusters/differential_expres...


,Barcode,Cluster
0,AAACATACAACCAC-1,2
1,AAACATTGAGCTAC-1,8
2,AAACATTGATCAGC-1,2
3,AAACCGTGCTTCCG-1,3
4,AAACCGTGTATGCG-1,1
5,AAACGCACTGGTAC-1,2
6,AAACGCTGACCAGT-1,2
7,AAACGCTGGTTCTT-1,2
8,AAACGCTGTAGCCA-1,2
9,AAACGCTGTTTCTG-1,10


## Recommended starting point for analysis with Scanpy

For clustering, visualization, and cell-type exploration, start with the filtered count matrix rather than BAM/H5. This loads all cells and genes into an `AnnData` object while retaining sparse storage.

In [10]:
import scanpy as sc

adata = sc.read_10x_mtx(FILTERED, var_names="gene_symbols", cache=False)
adata.var_names_make_unique()
print(adata)

# A standard exploratory workflow. Run once when you are ready to analyze.
# sc.pp.filter_cells(adata, min_genes=200)
# sc.pp.filter_genes(adata, min_cells=3)
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
# adata = adata[:, adata.var.highly_variable].copy()
# sc.pp.scale(adata, max_value=10)
# sc.tl.pca(adata)
# sc.pp.neighbors(adata)
# sc.tl.umap(adata)
# sc.pl.umap(adata)

AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'


When finished, close the two open binary-file handles:

```python
bam.close()
h5.close()
```